# 🔬 Entrenamiento YOLO11 (8 Equipos Oficiales) - Bromatología UTEQ
### Proyecto: Asistente Inteligente de Laboratorio con Visión Artificial y Gemini 2.5 Flash
**Universidad Técnica Estatal de Quevedo (UTEQ)**  
**Facultad de Ciencias Pecuarias y Biológicas**  
*Carrera de Ingeniería en Alimentos / Zootecnia*

---

### 📋 Lista de los 8 Equipos Oficiales Seleccionados:
1. **Sistema de Tratamiento y Desionizacion deAgua** (Sistema Multietapa con Manómetros)
2. **Viscosímetro Brookfield Modelo DV-E** (AMETEK Brookfield DV-E)
3. **Destilador de Agua Continuo Metalico** (Destilador Mural en Acero Inoxidable 4 L/h)
4. **Analizador de Fibra Cruda y Fracciones** (J.P. SELECTA Dosi-Fiber)
5. **Bomba de Vacio por Recirculacion de Agua** (J.P. SELECTA Cat. 4001611)
6. **Microscopio Trinocular** (Carl Zeiss Primo Star Trinocular)
7. **destilación por arrastre de vapor** (J.P. SELECTA Pro-Nitro)
8. **Destilador de proteina** (Fisher Scientific Distillation Unit 100)

---

Este cuaderno entrena el modelo de detección **YOLO11 Nano (`yolo11n.pt`)** con aceleración por GPU T4 y lo exporta a **TensorFlow Lite (`.tflite`)** listo para ejecutarse en tiempo real en la cámara de la app móvil Android.

## 🛠️ Paso 1: Configurar GPU y Dependencias de Ultralytics
Asegúrate de que en Colab esté seleccionada la **GPU T4** en:
`Entorno de ejecución > Cambiar tipo de entorno de ejecución > Acelerador de hardware > T4 GPU`.

In [ ]:
# 1. Verificar disponibilidad de GPU
!nvidia-smi

# 2. Instalar Ultralytics y dependencias para exportación TFLite
!pip install -q --upgrade ultralytics
!pip install -q tensorflow

import ultralytics
ultralytics.checks()

## 📦 Paso 2: Cargar el Dataset de los 8 Equipos desde Roboflow
Exporta tu dataset desde **Roboflow** en formato **YOLOv11** y súbelo como archivo `.zip` ejecutando la siguiente celda.

In [ ]:
import os
import glob
import shutil
import zipfile
import yaml
from google.colab import files

# Limpieza de ejecuciones anteriores
!rm -rf dataset runs *.zip *.tflite

print("📥 Selecciona el archivo .zip de Roboflow con los 8 equipos:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

print(f"📦 Descomprimiendo {zip_name}...")
os.makedirs('dataset', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('dataset')

# Localizar archivo data.yaml generado por Roboflow
yaml_files = glob.glob('dataset/**/data.yaml', recursive=True)
if not yaml_files:
    raise FileNotFoundError("❌ No se encontró data.yaml en el archivo ZIP subido.")

data_yaml_path = yaml_files[0]
base_dir = os.path.dirname(data_yaml_path)
print(f"✅ data.yaml ubicado en: {data_yaml_path}")

# Leer y validar clases en data.yaml
with open(data_yaml_path, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

classes = data_cfg.get('names', [])
print(f"\n🎯 Número total de clases detectadas: {len(classes)}")
print("📋 Clases del entrenamiento:")
for idx, cls_name in enumerate(classes):
    print(f"  [{idx}] {cls_name}")

# Ajustar rutas relativas para el entrenamiento en Colab
data_cfg['train'] = os.path.join(base_dir, 'train', 'images')
data_cfg['val'] = os.path.join(base_dir, 'valid', 'images')
if os.path.exists(os.path.join(base_dir, 'test', 'images')):
    data_cfg['test'] = os.path.join(base_dir, 'test', 'images')

with open(data_yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

# Generar automáticamente el archivo labels.txt sincronizado con la app Android
with open('labels.txt', 'w', encoding='utf-8') as f:
    for cls_name in classes:
        f.write(f"{cls_name}\n")

print("\n✅ Archivo labels.txt sincronizado correctamente para Android.")

## 🚀 Paso 3: Entrenar YOLO11 Nano (`yolo11n.pt`)
Entrenaremos con **YOLO11 Nano** durante **100 épocas** (con *early stopping* a las 25 épocas sin mejora) y aumentación de datos adaptada a las condiciones de iluminación de laboratorio.

In [ ]:
from ultralytics import YOLO

# Cargar modelo base YOLO11 Nano preentrenado
model = YOLO('yolo11n.pt')

# Iniciar entrenamiento
results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=25,
    save=True,
    device=0,
    project='yolo11_uteq_8_equipos',
    name='entrenamiento_oficial',
    exist_ok=True,
    # Aumentaciones balanceadas para equipos de laboratorio
    mosaic=1.0,
    degrees=5.0,
    scale=0.3,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4
)

print("\n🎉 ¡Entrenamiento completado exitosamente!")

## 📊 Paso 4: Evaluar Métricas del Modelo (mAP@0.5 y Confusión)
Calcula la precisión media del modelo sobre el conjunto de validación.

In [ ]:
from IPython.display import Image, display

# Validar mejor modelo
metrics = model.val()
print(f"\n📈 Precisión Media (mAP@0.5): {metrics.box.map50 * 100:.2f}%")
print(f"📈 Precisión Estricta (mAP@0.5:0.95): {metrics.box.map * 100:.2f}%")

# Mostrar matriz de confusión si fue generada
confusion_matrix_path = 'yolo11_uteq_8_equipos/entrenamiento_oficial/confusion_matrix.png'
if os.path.exists(confusion_matrix_path):
    print("\n📊 Matriz de Confusión:")
    display(Image(filename=confusion_matrix_path))

## 📱 Paso 5: Exportar a TensorFlow Lite (`.tflite`) y Descargar Paquete para Android
Esta celda exporta los pesos entrenados a TensorFlow Lite optimizado para dispositivos móviles y genera un paquete `.zip` que contiene:
1. `yolo11_bromatologia.tflite` (Modelo de visión offline para la cámara)
2. `labels.txt` (Las etiquetas oficiales en el orden exacto del modelo)

In [ ]:
import glob
import shutil
import zipfile
from google.colab import files
from ultralytics import YOLO

# 1. Localizar los mejores pesos (best.pt)
best_weights = 'yolo11_uteq_8_equipos/entrenamiento_oficial/weights/best.pt'
if not os.path.exists(best_weights):
    found = glob.glob('yolo11_uteq_8_equipos/**/weights/best.pt', recursive=True)
    if found:
        best_weights = found[0]
    else:
        raise FileNotFoundError("❌ No se encontró el archivo best.pt")

print(f"⚙️ Cargando mejores pesos para exportación: {best_weights}")
trained_model = YOLO(best_weights)

# 2. Exportar a formato TensorFlow Lite (.tflite)
print("⚙️ Exportando modelo a TensorFlow Lite (imgsz=640)...")
exported_result = trained_model.export(format='tflite', imgsz=640)

# 3. Localizar el archivo .tflite generado y renombrarlo al nombre oficial de la app
tflite_candidates = glob.glob('yolo11_uteq_8_equipos/**/*.tflite', recursive=True) + glob.glob('**/*.tflite', recursive=True)
if not tflite_candidates:
    raise FileNotFoundError("❌ No se generó el archivo .tflite durante la exportación.")

official_tflite = 'yolo11_bromatologia.tflite'
shutil.copyfile(tflite_candidates[0], official_tflite)
tflite_size_mb = os.path.getsize(official_tflite) / (1024 * 1024)
print(f"\n✅ Modelo TFLite generado con éxito: {official_tflite} ({tflite_size_mb:.2f} MB)")

# 4. Empaquetar modelo y etiquetas en un solo ZIP para la app Android
bundle_zip = 'modelo_8_equipos_uteq.zip'
with zipfile.ZipFile(bundle_zip, 'w') as z:
    z.write(official_tflite)
    z.write('labels.txt')

print(f"\n📦 Paquete creado: {bundle_zip}")
print("📥 Iniciando descarga automática...")
files.download(bundle_zip)

print("\n💡 INSTRUCCIONES PARA TU PROYECTO:")
print("1. Descomprime modelo_8_equipos_uteq.zip en tu computadora.")
print("2. Copia 'yolo11_bromatologia.tflite' y 'labels.txt' en la carpeta:")
print("   DetectorDeMaterialesLaboratorio/app/src/main/assets/")
print("3. ¡Listo! La app detectará los 8 equipos en tiempo real.")